# GARCH(1,1): the Fiorentini-Calzolari-Panattoni benchmark

Does Julia's GARCH(1,1) reach the benchmark estimates? Fiorentini, Calzolari and Panattoni (1996) computed, with analytic derivatives, the maximum likelihood estimates of a GARCH(1,1) model with normal errors and a constant mean on `bg96`, the 1974 daily Deutsche mark / British pound returns of Bollerslev and Ghysels (1996). Brooks, Burke and Persand (2001, Section 2) quote the estimates to 6 significant figures, and found that only one of the nine packages they tested reproduced them with its default settings (Section 4).

A GARCH model estimated by conditional maximum likelihood is defined by its start-up as well as by its equations: the squared residual and the variance of the day before the first one. The benchmark sets both to the mean squared residual, recomputed at each value of the mean (Brooks, Burke and Persand 2001, Section 2, equations 1-4). The model is fitted twice: with ARCHModels' `fit`, the code of the GARCH tab of GARCH-Type Models, and with the benchmark's own likelihood, written out below and maximised with Optim from two starting values.

The number of significant digits an estimate shares with the published one is the log relative error, the measure of Brooks, Burke and Persand (2001, Section 4) and of McCullough (1998, *The American Statistician* 52(4), 358-366): -log10(|estimate - published| / |published|), rounded down. The values are published to 6 significant figures, so 6 is the most an estimate can show.

In [ ]:
Pollis.packages("ARCHModels", "Optim", "PollisDatasets")

## Set-up

`digits_same` counts the significant digits an estimate shares with the published value, `report` prints each published value with the estimates below it, and `summary` the fewest digits of each fit.

In [ ]:
using ARCHModels, Optim, PollisDatasets, Statistics

using Printf

"""
	digits_same(value, published)

The number of significant digits `value` shares with `published`: the log relative error,
rounded down, at most 6, the figures published.
"""
function digits_same(value, published)
	isfinite(value) || return 0
	error = abs(value - published) / abs(published)
	return error == 0 ? 6 : clamp(floor(Int, -log10(error)), 0, 6)
end

"""
	report(columns, rows)

Print each published value with the values of `columns` below it, and the significant digits
each shares with it. Each row is `(label, published, values...)`, one value for each column.

Gives: the fewest digits shared in each column.
"""
function report(columns, rows)
	fewest = fill(6, length(columns))
	for (label, p, values...) in rows
		@printf("%-8s%-22s%18.10g\n", label, "published", p)
		for (k, (column, v)) in enumerate(zip(columns, values))
			d = digits_same(v, p)
			fewest[k] = min(fewest[k], d)
			@printf("%-8s%-22s%18.10g  %s\n", "", column, v, d == 0 ? "DIFFERENT" : "same to $d significant digits")
		end
	end
	return fewest
end

"""
	summary(columns, fewest)

Print the fewest significant digits shared with the published values by each of `columns`.
"""
function summary(columns, fewest)
	for (column, d) in zip(columns, fewest)
		@printf("%-22s%s\n", column, d == 0 ? "DIFFERENT" : "$d significant digits")
	end
end

## 1. The dataset and the published estimates

The returns of `bg96`, and the estimates of Fiorentini, Calzolari and Panattoni (1996) as Brooks, Burke and Persand (2001, Section 2) quote them, in the order mu, omega, alpha, beta.

In [ ]:
returns = Float64.(collect(skipmissing(dataset("bg96").return)))
parameters = ["mu", "omega", "alpha", "beta"]
published = [-0.00619041, 0.0107613, 0.153134, 0.805974]
length(returns)

## 2. Fit with ARCHModels

`fit(GARCH{1, 1}, returns)` with its defaults: normal errors and a constant mean. ARCHModels sets the first day's variance to the sample variance of the returns, fixed before the estimation, and gives its estimates in the order omega, beta, alpha, mu.

In [ ]:
model = fit(GARCH{1, 1}, returns)
package = coef(model)[[4, 1, 3, 2]]  # mu, omega, alpha, beta

## 3. Fit the benchmark's likelihood

The negative log likelihood of the benchmark (Brooks, Burke and Persand 2001, Section 2, equations 1-4): the residual of each day is its return less mu, its variance is omega plus alpha times the previous squared residual plus beta times the previous variance, and the previous squared residual and variance of the first day are both the mean squared residual at the current mu. Optim's BFGS maximises it, with derivatives by automatic differentiation (ForwardDiff), until the gradient is below `g_tol = 1e-12`: from ARCHModels' estimates (start 1) and from a point far from them (start 2). A variance that is not positive gives an infinite value, which the line search steps back from.

In [ ]:
"""
	negloglik(theta, returns)

The negative log likelihood of a GARCH(1,1) model with normal errors and a constant mean, with the
start-up of the benchmark. `theta` is `[mu, omega, alpha, beta]`.
"""
function negloglik(theta, returns)
	mu, omega, alpha, beta = theta
	e = returns .- mu
	h = mean(abs2, e)  # the variance of the day before the first one
	e2 = h             # and its squared residual
	total = zero(eltype(theta))
	for t in eachindex(e)
		h = omega + alpha * e2 + beta * h
		h > 0 || return oftype(total, Inf)
		total += (log(h) + e[t]^2 / h + log(2pi)) / 2
		e2 = e[t]^2
	end
	return total
end

"""
	benchmark_fit(returns, start)

Maximise the benchmark's likelihood from `start` with BFGS.

Gives: the estimates in the order mu, omega, alpha, beta, and whether Optim reports convergence.
"""
function benchmark_fit(returns, start)
	result = optimize(theta -> negloglik(theta, returns), start, BFGS(), Optim.Options(g_tol = 1e-12); autodiff = :forward)
	return (coef = Optim.minimizer(result), converged = Optim.converged(result))
end

fits = [benchmark_fit(returns, package), benchmark_fit(returns, [0.0, 0.05, 0.2, 0.5])]
columns = ["ARCHModels", "benchmark, start 1", "benchmark, start 2"]
rows = [(name, published[i], package[i], (f.coef[i] for f in fits)...) for (i, name) in enumerate(parameters)]
fewest = report(columns, rows)
println("converged  ", join(("$column: $(f.converged)" for (column, f) in zip(columns[2:end], fits)), ", "))

## 4. Summary

The fewest significant digits each fit shares with the published estimates.

In [ ]:
summary(columns, fewest)

## What the summary shows

ARCHModels shares 2 to 4 significant digits with the published estimates: 2 for mu and alpha, 3 for beta and 4 for omega. Its estimates are not wrong: they are those of a slightly different model, whose first day's variance is the sample variance of the returns. On these 1974 days the start-up moves mu and alpha in their third significant digit: -0.0061664 against -0.0061904, and 0.15341 against 0.15313.

The benchmark's likelihood, maximised from either start, gives the same estimates to 10 digits. They share 6 significant digits with the published alpha and beta, and 5 with mu and omega, whose sixth digits differ by one: -0.00619040 against -0.00619041, and 0.0107614 against 0.0107613.

So Julia reaches the benchmark when it estimates the benchmark's model, and ARCHModels with its defaults, like most of the packages Brooks, Burke and Persand (2001) tested, does not. State the start-up when you report GARCH estimates, and compare packages on the same start-up. GARCH-Type Models, in the Finance toolbox, has a tutorial, Replicate Published Results, that also writes ARCHModels' start-up out and finds ARCHModels' estimates with it.